# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maheen-armghan/flyrank-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [4]:
import os
if not os.path.exists("flyrank-internship"):
    !git clone https://github.com/maheen-armghan/flyrank-internship.git
os.chdir("flyrank-internship")
print("Now in:", os.getcwd())

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 358, done.
remote: Counting objects: 100% (358/358), done.
remote: Compressing objects: 100% (174/174), done.
remote: Total 358 (delta 200), reused 293 (delta 156), pack-reused 0 (from 0)
Receiving objects: 100% (358/358), 1.94 MiB | 14.42 MiB/s, done.
Resolving deltas: 100% (200/200), done.
Now in: /content/flyrank-internship/flyrank-internship


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Reload and prep data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

features = ["search_volume", "competition", "cpc", "word_count", "content_age_days",
            "impressions_90d", "sessions_90d", "avg_position", "ctr",
            "days_since_last_update", "engagement_rate", "scroll_rate"]
features = [f for f in features if f in df.columns]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"]
groups = df["client_id"] if "client_id" in df.columns else df["content_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = df.iloc[test_idx].copy()

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

# Fair baseline
test_df["fair_baseline_score"] = (
    (test_df["days_since_last_update"] >= 180).astype(int) *
    (test_df["impressions_90d"] >= 500).astype(int) *
    test_df["impressions_90d"]
)
fair_baseline_p50 = precision_at_k(test_df["fair_baseline_score"].values, y_test.values, 50)

# Models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(max_depth=4, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=6, class_weight="balanced", random_state=42),
}

results_fixed = {"Baseline (fair, no leakage)": fair_baseline_p50}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    results_fixed[name] = precision_at_k(proba, y_test.values, 50)

results_table_fixed = pd.DataFrame(results_fixed.items(), columns=["Method", "Precision@50"]).sort_values("Precision@50", ascending=False)
print(results_table_fixed.to_string(index=False))

/usr/local/lib/python3.13/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


                     Method  Precision@50
Baseline (fair, no leakage)          0.64
        Logistic Regression          0.60
              Decision Tree          0.60
              Random Forest          0.58


**Finding 1:** [state the paper's claim, e.g. "Pages with X characteristic show Y% higher engagement"]
**Methodology question:** Where does the label/outcome measure come from — is it a future observed outcome or a current-window proxy? Does the validation design (random split vs. grouped vs. time-aware) match the kind of claim being made?

**Finding 2:** [state a second claim]
**Methodology question:** [tailored question — e.g. "Does this correlation control for position/tier the way Signal 2 in my own baseline work required?" or "Is this an observed association or is it being presented as causal?"]

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**Before:** My Week 5 baseline initially scored a suspicious Precision@50 of 1.00 because the score formula used `is_declining_label` directly — checking the label against itself. This was invalid, leakage-driven result.

**After:** Rebuilding the baseline using only observable pre-decision signals (staleness x visibility, no label input), under the same client-grouped holdout split as the trained models, gave Precision@50 = 0.64. This is notably higher than every trained model: Logistic Regression (0.60), Decision Tree (0.60), and Random Forest (0.58).

**What this means:** once leakage was removed, the simple two-condition rule (stale AND visible) outperformed every model I trained. This is a genuine, if humbling, finding — not every lane's data guarantees a model beats a hand rule, and reporting that honestly matters more than forcing a "model wins" narrative. A likely explanation is that my baseline's two conditions (staleness >= 180 days, impressions >= 500) happen to align well with the specific 50-page top slice being evaluated, while the models were trained to optimize probability calibration broadly rather than specifically for the top-50 cutoff. A next step worth trying: tune the models' decision threshold specifically for Precision@50 rather than using the default probability ranking, or add interaction features (e.g. staleness x impressions) that let the models directly capture what the baseline's AND logic already captures.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(results_table_fixed.to_string(index=False))

                     Method  Precision@50
Baseline (fair, no leakage)          0.64
        Logistic Regression          0.60
              Decision Tree          0.60
              Random Forest          0.58


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Leakage found:** the original Week 5 baseline scoring formula included `is_declining_label` as a multiplicative term — the target leaking directly into the "feature." This inflated Precision@50 to a meaningless 1.00.

**Leakage checked and cleared:** features used by the trained models (impressions_90d, avg_position, content_age_days, etc.) were confirmed to be observable pre-decision signals only, per the w03 data contract's feature/label/context/excluded classification — none are derived from `trend_direction` or `is_declining_label`.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm no label-derived columns are in the model's feature set
suspect_cols = [c for c in features if "declin" in c.lower() or "trend" in c.lower() or "label" in c.lower()]
print("Suspect leakage columns in feature set:", suspect_cols if suspect_cols else "None found — clean.")

Suspect leakage columns in feature set: None found — clean.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Reviewing my own prior claims: in w04_baseline_score, I described CTR-vs-position as "CONFIRMED" this is an observed, directional pattern in this dataset, not proof of a universal SEO law, so I'm restating it as: "in this dataset, CTR is observed to drop consistently as position tier worsens." In w05_model, I initially reported Precision@50 = 1.00 for my baseline — this claim is now retracted and replaced with the fair, leakage-free comparison. All model results are decision-support for prioritizing review, not proof that any specific page will decline or recover.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.